# Searching a large, growing archive, with `Streaming Hybrid Index`

In [1]:
%%capture
!pip install datasets sentence-transformers simlar simlar-engine ipywidgets

## Load the archive

We use [`fancyzhx/ag_news`](https://huggingface.co/datasets/fancyzhx/ag_news) — 120,000 news articles across four categories. We take 5,000 and feed them in as five batches of 1,000, simulating a stream. (Bump these numbers up and the same code keeps working — the point is that the corpus never has to arrive, or be embedded, all at once.)

In [ ]:
# optional
from huggingface_hub import login
login(token="<your_token>")

In [3]:
from datasets import load_dataset

LABELS = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}

ds = load_dataset("fancyzhx/ag_news", split="train")
CORPUS = [str(t) for t in ds["text"]]        # streaming search returns positions INTO this list
CORPUS_SIZE = len(CORPUS)
CATEGORY = [LABELS[l] for l in ds["label"]]

print(f"Loaded {len(CORPUS)} articles")
print(f"Sample: {CORPUS[0][:90]}")

Loaded 120000 articles
Sample: Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's d


## Embed the catalog

In [4]:
import numpy as np
import time
from sentence_transformers import SentenceTransformer
from simlar import StreamingHybridIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

index = StreamingHybridIndex(top_k=10)

BATCH = 20000
for start in range(0, len(CORPUS), BATCH):
    batch_texts = CORPUS[start:start + BATCH]
    batch_vecs = model.encode(batch_texts, normalize_embeddings=True).astype(np.float32)
    start_time = time.time()
    index.add_batch(batch_texts, batch_vecs)
    end_time = time.time()
    print(f"  ingested batch {start // BATCH + 1}: rows {start}-{start + len(batch_texts) - 1}")
    print(f"    took {end_time - start_time:.2f} seconds")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Split strings:   0%|          | 0/20000 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/20000 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/20000 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/20000 [00:00<?, ?it/s]

...

Split strings:   0%|          | 0/20000 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/20000 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/20000 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/20000 [00:00<?, ?it/s]

  ingested batch 6: rows 100000-119999
    took 0.45 seconds


## Search across all shards

`search(query_text, query_vector, k)` searches every shard and merges globally. Its
return contract differs from the other indexes: it returns **`(ids, scores)` NumPy
arrays**, where each `id` is an **integer position** into the corpus you ingested — look
the document up with `CORPUS[int(id)]`. Scores are fused relevance, so **higher is
better**.

In [5]:
def search_new(query_text):
    query_vec = model.encode([query_text], normalize_embeddings=True).astype(np.float32)[0]
    ids, scores = index.search(query_text, query_vec, k=5)

    print(f"\nQuery: {query_text!r}\n")
    for pos, score in zip(ids, scores):
        pos = int(pos)
        print(f"  pos={pos:5d}  score={score:.4f}  [{CATEGORY[pos]}]  {CORPUS[pos][:70]}")

In [6]:
%%time
search_new("stock market rally and corporate earnings")


Query: 'stock market rally and corporate earnings'

  pos=88452  score=0.3410  [Business]  Stock markets rally on Fed statement, lower oil prices By George Chamb
  pos= 9786  score=0.2504  [World]  Stocks Rally on Lower Oil Prices NEW YORK - Stocks rallied in quiet tr
  pos=59557  score=0.2007  [World]  Stocks Mixed on Strong Earnings Reports NEW YORK - Falling oil prices 
  pos=75096  score=0.1676  [Business]  Stocks Rally on Oil, Economic News With oil prices dropping, stocks ra
  pos=24461  score=0.1456  [Business]  Earning Reports Keep Investors on Edge  NEW YORK (Reuters) - Investors
CPU times: user 3.97 s, sys: 159 ms, total: 4.13 s
Wall time: 3.9 s


## Add more batches to a live index

The index keeps growing — later batches become new shards and are searched alongside the
old ones, with no rebuild. (Here we simply confirm the shard count; in production this is
the next slice of the stream arriving.)

In [7]:
extra_texts = [
    "Central bank raises interest rates amid inflation concerns",
    "Tech startup unveils new AI chip for data centers",
]
extra_vecs = model.encode(extra_texts, normalize_embeddings=True).astype(np.float32)

# NOTE: positions are relative to the corpus list you maintain — extend it in lockstep.
CORPUS.extend(extra_texts)
CATEGORY.extend(["Business", "Sci/Tech"])
index.add_batch(extra_texts, extra_vecs)

Split strings:   0%|          | 0/2 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/2 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/2 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/2 [00:00<?, ?it/s]

In [8]:
%%time
search_new("interest rate hike by the central bank")
search_new("new chips for data centers")


Query: 'interest rate hike by the central bank'

  pos=120000  score=0.3333  [Business]  Central bank raises interest rates amid inflation concerns
  pos=120000  score=0.2500  [Business]  Central bank raises interest rates amid inflation concerns
  pos=120000  score=0.2000  [Business]  Central bank raises interest rates amid inflation concerns
  pos=120000  score=0.1667  [Business]  Central bank raises interest rates amid inflation concerns
  pos=120000  score=0.1429  [Business]  Central bank raises interest rates amid inflation concerns

Query: 'new chips for data centers'

  pos=120001  score=0.3333  [Sci/Tech]  Tech startup unveils new AI chip for data centers
  pos= 3926  score=0.2505  [Sci/Tech]  Intel Chips In for New Gateway PCs Desktops will be available at sever
  pos=102218  score=0.2001  [Business]  Chip sales up 1.5 percent in October Organizations are replacing aging
  pos=80975  score=0.1667  [Sci/Tech]  Intel devising chip line for consumer electronics com November 3, 2